In [ ]:
%env PROJ=/path/to/repo
%env PYTHONPATH=/path/to/repo:$PYTHONPATH
%env HF_HOME=/path/to/repo/tmp/huggingface
%env WANDB_MODE=online
%env EINV_CONFIG=layer/setup25.yaml
%env TRAIN_TAG=exp05

In [ ]:
from utils import p_args, gh
from utils.models import load_pretrained_model
from huggingface_hub import login

login(token=p_args.settings.hf_token)

# Load models
tokenizer, model, context_len = load_pretrained_model(
    model_type='einv',
    resume_from=True, # Set False for training
    device_map=gh.device
)

In [ ]:
import os
import torch
PROJ = os.getenv('PROJ')
_dir = os.path.join(PROJ, 'vectors')
files = os.listdir(_dir)
concepts = {f[:-6]: f for f in files}

norm_vecs = {}

hs_dir = "/path/to/hidden_states/l3-8b/steer"
for concept in concepts:
    concepts[concept] = torch.load(os.path.join(_dir, concepts[concept]))
    _file = os.path.join(hs_dir, f"{concept}.pt")
    if not os.path.exists(_file):
        print(f"====================== {concept}: missing =================")
        print(f"skipping...")
        continue    
    
    hs = torch.load(os.path.join(hs_dir, f"{concept}.pt"))[25]
    print(f"====================== {concept}: {len(hs)} =================")
    hs_norm = hs.norm(dim=1).mean()
    vec_norm = concepts[concept].norm()
    print(vec_norm, hs_norm)
    norm_vecs[concept] = concepts[concept] / vec_norm * hs_norm

In [ ]:
# decode diff mean vector for each concept
import os
from utils.core.files import read_file, read_json
from utils.models.constants import DEFAULT_EM_TOKEN
import torch
PROJ = os.getenv('PROJ')
sys_prompt = read_file(os.path.join(PROJ, 'datasets/prompts', 'sys_prompt.txt'))[0]
user_prompt = read_json(os.path.join(PROJ, 'datasets/prompts', 'prompts.json'))[0]

em_tokens = DEFAULT_EM_TOKEN * p_args.proj.n_vecs
bos = tokenizer.bos_token if tokenizer.bos_token is not None else ""

prefix_str = f"{bos}{em_tokens}\n{sys_prompt}\n{user_prompt}\nAnswer:"   # BEFORE the answer

input_ids = torch.tensor(tokenizer.encode(prefix_str, add_special_tokens=False)).unsqueeze(0).to(gh.device)
attn_mask = input_ids.ne(tokenizer.pad_token_id).to(gh.device)

In [ ]:
input_len = input_ids.shape[1]
for concept, vec in norm_vecs.items():
    print(f"===================Decoding concept: {concept}====================")
    generated = None
    # print(vec.shape)
    with torch.no_grad():
        generated = model.generate(
            input_ids=input_ids,
            hidden_embeds=vec.to(gh.device),
            attention_mask=attn_mask,
            do_sample=False,
            num_beams=1,
            max_new_tokens=p_args.data.max_len,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )

    output_str = tokenizer.decode(generated[0, input_len:], skip_special_tokens=True)
    print(f"Output:\n{output_str}\n\n")

In [ ]:
input_text = "Commission (1945) was an organisation whose dissolution was by the Soviet Union's"
from utils.data.embeddings import _get_embeddings
from transformer_lens import HookedTransformer
import torch

torch.cuda.empty_cache()
tokens = tokenizer.tokenize(input_text)
cache_model = HookedTransformer.from_pretrained(
        p_args.model.model_id, 
        device=gh.device, 
        n_devices=1,
        cache_dir=p_args.path.weight_dir
    )
hs = _get_embeddings(tokens, cache_model)
del cache_model
torch.cuda.empty_cache()

In [ ]:
from sae_lens import SAE
import torch as th

th.cuda.empty_cache()

release = "llama_scope_lxa_32x"
sae_id = "l25a_32x"
sae = SAE.from_pretrained(release, sae_id)

In [ ]:
for i in range(20):
    emb = (sae.W_dec[i] * 28.0 / sae.W_dec[i].norm()).unsqueeze(0)

    input_len = input_ids.shape[1]
    print(f"===================Decoding SAE latent {i}====================")
    generated = None
    # print(vec.shape)
    with torch.no_grad():
        generated = model.generate(
            input_ids=input_ids,
            hidden_embeds=emb.to(gh.device),
            attention_mask=attn_mask,
            do_sample=False,
            num_beams=1,
            max_new_tokens=p_args.data.max_len,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )

    output_str = tokenizer.decode(generated[0, input_len:], skip_special_tokens=True)
    print(f"Output:\n{output_str}\n\n")